In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import tensorflow as tf
import os
import tensorflow_hub as hub
import shutil
import string
import re

2026-03-20 10:07:52.028038: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 10:07:53.285925: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 10:07:57.485502: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
print(tf.__version__)
tf.config.list_physical_devices("GPU")

2.20.0


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
dataset_dir="./aclImdb_v1"

In [4]:
train_dir=os.path.join(dataset_dir,"aclImdb","train")
test_dir=os.path.join(dataset_dir,"aclImdb","test")
os.listdir(train_dir)

['labeledBow.feat',
 'neg',
 'pos',
 'unsupBow.feat',
 'urls_neg.txt',
 'urls_pos.txt',
 'urls_unsup.txt']

In [5]:
sample=os.path.join(train_dir,"pos","1181_9.txt")
with open(sample) as f:
    print(f.read())

Rachel Griffiths writes and directs this award winning short film. A heartwarming story about coping with grief and cherishing the memory of those we've loved and lost. Although, only 15 minutes long, Griffiths manages to capture so much emotion and truth onto film in the short space of time. Bud Tingwell gives a touching performance as Will, a widower struggling to cope with his wife's death. Will is confronted by the harsh reality of loneliness and helplessness as he proceeds to take care of Ruth's pet cow, Tulip. The film displays the grief and responsibility one feels for those they have loved and lost. Good cinematography, great direction, and superbly acted. It will bring tears to all those who have lost a loved one, and survived.


In [6]:
batch_size=32
rw_train_ds=tf.keras.utils.text_dataset_from_directory(train_dir,batch_size=batch_size,seed=42,validation_split=0.2,subset="training")

Found 25000 files belonging to 2 classes.
Using 20000 files for training.


I0000 00:00:1774001284.208794     811 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [7]:
rw_val_ds=tf.keras.utils.text_dataset_from_directory(train_dir,validation_split=0.2,seed=42,subset="validation",batch_size=batch_size)

Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


In [8]:
for text_batch,label_batch in rw_train_ds.take(1):
    for i in range(3):
        print(text_batch.numpy()[i])
        print(label_batch.numpy()[i])

b'"Pandemonium" is a horror movie spoof that comes off more stupid than funny. Believe me when I tell you, I love comedies. Especially comedy spoofs. "Airplane", "The Naked Gun" trilogy, "Blazing Saddles", "High Anxiety", and "Spaceballs" are some of my favorite comedies that spoof a particular genre. "Pandemonium" is not up there with those films. Most of the scenes in this movie had me sitting there in stunned silence because the movie wasn\'t all that funny. There are a few laughs in the film, but when you watch a comedy, you expect to laugh a lot more than a few times and that\'s all this film has going for it. Geez, "Scream" had more laughs than this film and that was more of a horror film. How bizarre is that?<br /><br />*1/2 (out of four)'
0
b"David Mamet is a very interesting and a very un-equal director. His first movie 'House of Games' was the one I liked best, and it set a series of films with characters whose perspective of life changes as they get into complicated situatio

2026-03-20 10:08:08.071834: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
print(" Label 0 corresponds to:",rw_train_ds.class_names[0])

 Label 0 corresponds to: neg


In [10]:
rw_test_ds=tf.keras.utils.text_dataset_from_directory(test_dir,batch_size=batch_size)

Found 25000 files belonging to 2 classes.


In [11]:
def custom_standardization(input_data):
    lower=tf.strings.lower(input_data)
    stripped_html=tf.strings.regex_replace(lower,"<br />"," ")
    return tf.strings.regex_replace(stripped_html,
                                    "[%s]" % re.escape(string.punctuation),
                                    "")

In [ ]:
re.escape(string.punctuation)
'!"\\#\\$%\\&\'\\(\\)\\*\\+,\\-\\./:;<=>\\?@\\[\\\\\\]\\^_`\\{\\|\\}\\~'

'!"\\#\\$%\\&\'\\(\\)\\*\\+,\\-\\./:;<=>\\?@\\[\\\\\\]\\^_`\\{\\|\\}\\~'

In [12]:
vectorized_layer=tf.keras.layers.TextVectorization(standardize=custom_standardization,
                                                   max_tokens=10000,
                                                   output_mode="int",
                                                   output_sequence_length=250)

In [13]:
train_text=rw_train_ds.map(lambda x,y:x)
vectorized_layer.adapt(train_text)

2026-03-20 10:08:23.125242: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [14]:
def vectorized_text(text,label):
    text=tf.expand_dims(text,-1)
    return vectorized_layer(text),label

In [15]:
text_batch,label_batch=next(iter(rw_train_ds))
print(text_batch[0])
print(rw_train_ds.class_names[label_batch[0]])
print(vectorized_text(text_batch[0],label_batch[0]))

tf.Tensor(b'Silent Night, Deadly Night 5 is the very last of the series, and like part 4, it\'s unrelated to the first three except by title and the fact that it\'s a Christmas-themed horror flick.<br /><br />Except to the oblivious, there\'s some obvious things going on here...Mickey Rooney plays a toymaker named Joe Petto and his creepy son\'s name is Pino. Ring a bell, anyone? Now, a little boy named Derek heard a knock at the door one evening, and opened it to find a present on the doorstep for him. Even though it said "don\'t open till Christmas", he begins to open it anyway but is stopped by his dad, who scolds him and sends him to bed, and opens the gift himself. Inside is a little red ball that sprouts Santa arms and a head, and proceeds to kill dad. Oops, maybe he should have left well-enough alone. Of course Derek is then traumatized by the incident since he watched it from the stairs, but he doesn\'t grow up to be some killer Santa, he just stops talking.<br /><br />There\'s

In [16]:
train_ds=rw_train_ds.map(vectorized_text)
val_ds=rw_val_ds.map(vectorized_text)
test_ds=rw_test_ds.map(vectorized_text)

In [17]:
autotune=tf.data.AUTOTUNE
train_ds=train_ds.cache().prefetch(buffer_size=autotune)
test_ds=test_ds.cache().prefetch(buffer_size=autotune)
val_ds=val_ds.cache().prefetch(buffer_size=autotune)

In [18]:
embedding_dim=16
model=tf.keras.Sequential([
    tf.keras.layers.Embedding(10000,embedding_dim),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1,activation="sigmoid")
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.compile(loss=tf.losses.BinaryCrossentropy(),
              optimizer="adam",
              metrics=[tf.metrics.BinaryAccuracy(threshold=0.5)])

In [20]:
epoches=10
history=model.fit(train_ds,validation_data=val_ds,epochs=epoches)

Epoch 1/10


2026-03-20 10:08:24.952249: I external/local_xla/xla/service/service.cc:163] XLA service 0x236f4370 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-20 10:08:24.952313: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-20 10:08:25.032358: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-20 10:08:25.334492: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92000


 13/625 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - binary_accuracy: 0.4491 - loss: 0.6940

I0000 00:00:1774001307.276306    1700 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 25ms/step - binary_accuracy: 0.6452 - loss: 0.6606 - val_binary_accuracy: 0.7402 - val_loss: 0.6081
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - binary_accuracy: 0.7814 - loss: 0.5437 - val_binary_accuracy: 0.8142 - val_loss: 0.4943
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - binary_accuracy: 0.8331 - loss: 0.4440 - val_binary_accuracy: 0.8316 - val_loss: 0.4261
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - binary_accuracy: 0.8558 - loss: 0.3812 - val_binary_accuracy: 0.8438 - val_loss: 0.3845
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - binary_accuracy: 0.8716 - loss: 0.3402 - val_binary_accuracy: 0.8434 - val_loss: 0.3632
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - binary_accuracy: 0.8823 - loss: 0.3098 - val_binary_accuracy: 0.8474 - val_loss: 0.3475
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - binary_accuracy: 0.8916 - loss: 0.2856 - val_binary_accuracy: 0.8582 - val_loss: 0.3305
Epoch 8/10
625/

In [23]:
loss , accuracy=model.evaluate(test_ds)
print(f'loss: {loss},accuracy:{accuracy}')

  1/782 ━━━━━━━━━━━━━━━━━━━━ 22s 28ms/step - binary_accuracy: 0.8438 - loss: 0.4276

782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8518 - loss: 0.3358
loss: 0.3357618451118469,accuracy:0.8518400192260742


In [27]:
export_model=tf.keras.Sequential([vectorized_layer,
                                  model])
export_model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),optimizer="adam",metrics=["accuracy"])
metrics=export_model.evaluate(rw_test_ds,return_dict=True)
print(metrics)

782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.8518 - loss: 0.3358
{'accuracy': 0.8518400192260742, 'loss': 0.3357617259025574}
